In [ ]:
# =====================================================
# PROJECT PATH SETUP
# =====================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

SCRIPTS_DIR = PROJECT_ROOT / "src"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

# =====================================================
# STANDARD LIBRARY
# =====================================================

from datetime import datetime
import time

# =====================================================
# THIRD-PARTY LIBRARIES
# =====================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

# =====================================================
# PROJECT CONFIGURATION
# =====================================================

from config import (
    MODEL_NAME,
    IMG_SIZE,
    NUM_CLASSES,
    LEARNING_RATES,
    SEEDS,
    DURATIONS,
    BATCH_SIZE,
    EPOCHS,
    GENRE_TO_INT
)

# =====================================================
# PROJECT UTILITIES
# =====================================================

from utils.paths import (
    get_output_paths,
    SPLIT_FILE,
    SPECTROGRAM_DIR
)

from utils.model_factory import (
    build_model
)

from utils.evaluation import (
    evaluate_model
)

# =====================================================
# SANITY CHECK
# =====================================================

print(f"Model: {MODEL_NAME}")
print(f"Seeds: {SEEDS}")
print(f"Durations: {DURATIONS}")

In [ ]:
model = build_model(
    model_name=MODEL_NAME,
    input_shape=IMG_SIZE + (3,),
    num_classes=NUM_CLASSES
)

model.summary()
print(
    model.optimizer.learning_rate.numpy()
)

In [ ]:
def get_split_dataframe():
    return pd.read_csv(SPLIT_FILE)

split_df = get_split_dataframe()


def build_file_lists(
        duration,
        split_name,
        split_df):

    split_df = (
        split_df[
            split_df["split"]
            == split_name
        ]
    )

    image_paths = []
    labels = []

    missing = []

    for _, row in split_df.iterrows():

        image_path = (
            SPECTROGRAM_DIR /
            "processed" /
            duration /
            row["genre"] /
            f"{row['file_id']}.png"
        )

        if image_path.exists():

            image_paths.append(
                str(image_path)
            )

            labels.append(
                GENRE_TO_INT[
                    row["genre"]
                ]
            )

        else:

            missing.append(str(image_path))

    print(
        f"Missing files: {len(missing)}"
    )

    for f in missing:
        print(f)

    return (
        image_paths,
        labels
    )

In [ ]:
paths, labels = build_file_lists(
    "15s",
    "train",
    split_df
)

print(len(paths))
print(len(labels))

print(paths[0])
print(labels[0])

In [ ]:
split_df = get_split_dataframe()

print(len(split_df))

print(split_df["split"].value_counts())

In [ ]:
def parse_image(
        image_path,
        label):

    image = tf.io.read_file(
        image_path
    )

    image = tf.image.decode_png(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = (
        tf.cast(
            image,
            tf.float32
        )
        / 255.0
    )

    return image, label


def make_dataset(
        paths,
        labels,
        training=False):

    ds = (
        tf.data.Dataset
        .from_tensor_slices(
            (paths, labels)
        )
    )

    ds = ds.map(
        parse_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:

        ds = ds.shuffle(
            buffer_size=len(paths),
            reshuffle_each_iteration=True
        )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


In [ ]:
train_paths, train_labels = (
    build_file_lists(
        "15s",
        "train",
        split_df
    )
)

train_ds = make_dataset(
    train_paths,
    train_labels,
    training=True
)

print(train_ds)

In [ ]:
sample_batch, _ = next(
    iter(train_ds)
)

preds = model.predict(
    sample_batch[:5],
    verbose=0
)

print(preds)

print(
    np.sum(preds, axis=1)
)

In [ ]:
for images, labels in train_ds.take(1):

    print(images.shape)
    print(labels.shape)

    print(images.dtype)
    print(labels.dtype)

    print(
        images.numpy().min(),
        images.numpy().max()
    )

In [ ]:
# =====================================================
# EXPERIMENT LOOP
# =====================================================

import time

paths = get_output_paths(MODEL_NAME)

split_df = get_split_dataframe()

all_summary_rows = []


def set_seed(seed):

    np.random.seed(seed)
    tf.random.set_seed(seed)


for seed in SEEDS:

    print("\n" + "=" * 60)
    print(f"Seed: {seed}")
    print("=" * 60)

    set_seed(seed)

    results = []
    summary_results = []

    for duration in DURATIONS:

        print(f"\nDuration: {duration}")

        # -----------------------------------------
        # Build datasets
        # -----------------------------------------

        train_files, train_labels = (
            build_file_lists(
                duration,
                "train",
                split_df
            )
        )

        val_files, val_labels = (
            build_file_lists(
                duration,
                "val",
                split_df
            )
        )

        test_files, test_labels = (
            build_file_lists(
                duration,
                "test",
                split_df
            )
        )

        train_ds = make_dataset(
            train_files,
            train_labels,
            training=True
        )

        val_ds = make_dataset(
            val_files,
            val_labels
        )

        test_ds = make_dataset(
            test_files,
            test_labels
        )

        # -----------------------------------------
        # Build model
        # -----------------------------------------

        learning_rate = LEARNING_RATES[MODEL_NAME]

        model = build_model(
            model_name=MODEL_NAME,
            input_shape=IMG_SIZE + (3,),
            num_classes=NUM_CLASSES,
            learning_rate=learning_rate
        )

        parameter_count = model.count_params()
        
        # -----------------------------------------
        # Save architecture metadata (once)
        # -----------------------------------------
        
        MODEL_FAMILY_MAP = {
        
            "cnn": "CNN",
        
            "mobilenetv2": "CNN",
        
            "vgg": "CNN",
        
            "shallowresnet": "ResNet",
        
            "residualcnn": "ResNet",
        
            "crnn": "Recurrent",
        
            "capsnet": "Capsule",
        
            "mlp": "Dense"
        }
        
        metadata_path = (
            paths["metadata"] /
            f"{MODEL_NAME}_metadata.csv"
        )
        
        if not metadata_path.exists():
        
            pd.DataFrame([{
        
                "model":
                    MODEL_NAME,
        
                "model_family":
                    MODEL_FAMILY_MAP.get(
                        MODEL_NAME.lower(),
                        "Unknown"
                    ),
        
                "parameter_count":
                    parameter_count,
        
                "input_height":
                    IMG_SIZE[0],
        
                "input_width":
                    IMG_SIZE[1],
        
                "channels":
                    3,
        
                "num_classes":
                    NUM_CLASSES,
        
                "initial_learning_rate":
                    learning_rate,
        
                # -----------------------------
                # Dataset metadata
                # -----------------------------
        
                "train_samples":
                    len(train_files),
        
                "val_samples":
                    len(val_files),
        
                "test_samples":
                    len(test_files)
        
            }]).to_csv(
                metadata_path,
                index=False
            )
            
        # -----------------------------------------
        # Callbacks
        # -----------------------------------------

        checkpoint_path = (
            paths["checkpoints"] /
            f"{MODEL_NAME}_{duration}_seed_{seed}.keras"
        )

        early_stop = EarlyStopping(
            monitor="val_accuracy",
            mode="max",
            patience=20,
            restore_best_weights=True,
            verbose=1
        )

        reduce_lr = ReduceLROnPlateau(
            monitor="val_accuracy",
            mode="max",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        )

        checkpoint = ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        )

        # -----------------------------------------
        # Train
        # -----------------------------------------

        start_time = time.time()

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            callbacks=[
                early_stop,
                reduce_lr,
                checkpoint
            ]
        )

        training_seconds = (
            time.time() - start_time
        )

        # -----------------------------------------
        # Best epoch (based on val_accuracy)
        # -----------------------------------------

        best_epoch = (
            np.argmax(
                history.history["val_accuracy"]
            ) + 1
        )

        # -----------------------------------------
        # Save enriched history
        # -----------------------------------------

        history_df = pd.DataFrame(
            history.history
        )

        history_df["epoch"] = np.arange(
            1,
            len(history_df) + 1
        )

        history_df["model"] = MODEL_NAME
        history_df["seed"] = seed
        history_df["duration"] = duration

        history_df["best_val_accuracy_so_far"] = (
            history_df["val_accuracy"].cummax()
        )

        history_df["best_val_loss_so_far"] = (
            history_df["val_loss"].cummin()
        )

        history_df["accuracy_gap"] = (
            history_df["accuracy"]
            -
            history_df["val_accuracy"]
        )

        history_df["loss_gap"] = (
            history_df["val_loss"]
            -
            history_df["loss"]
        )

        history_df.to_csv(
            paths["histories"] /
            f"training_history_{duration}_seed_{seed}.csv",
            index=False
        )

        # -----------------------------------------
        # Save final model
        # -----------------------------------------

        #model.save(
        #    paths["models"] /
        #    f"{MODEL_NAME}_{duration}_seed_{seed}_final.keras"
        #)

        # -----------------------------------------
        # Evaluate
        # -----------------------------------------

        metrics = evaluate_model(
            model=model,
            test_ds=test_ds,
            model_name=MODEL_NAME,
            seed=seed,
            duration=duration,
            best_epoch=best_epoch
        )

        # -----------------------------------------
        # Save confusion matrix
        # -----------------------------------------

        np.save(
            paths["confusion_matrices"] /
            f"confusion_matrix_{duration}_seed_{seed}.npy",
            metrics["confusion_matrix"]
        )

        # -----------------------------------------
        # Save per-genre metrics
        # -----------------------------------------

        genre_rows = []

        for genre_name, genre_index in GENRE_TO_INT.items():

            genre_metrics = (
                metrics["classification_report"]
                [str(genre_index)]
            )

            genre_rows.append({

                "model": MODEL_NAME,
                "seed": seed,
                "duration": duration,
                "genre": genre_name,

                "precision":
                    genre_metrics["precision"],

                "recall":
                    genre_metrics["recall"],

                "f1":
                    genre_metrics["f1-score"]
            })

        pd.DataFrame(
            genre_rows
        ).to_csv(
            paths["genre_results"] /
            f"genre_results_{duration}_seed_{seed}.csv",
            index=False
        )

        # -----------------------------------------
        # Summary row
        # -----------------------------------------

        best_val_accuracy = float(
            np.max(
                history.history["val_accuracy"]
            )
        )

        best_val_loss = float(
            np.min(
                history.history["val_loss"]
            )
        )

        final_train_accuracy = float(
            history.history["accuracy"][-1]
        )

        final_val_accuracy = float(
            history.history["val_accuracy"][-1]
        )

        final_train_loss = float(
            history.history["loss"][-1]
        )

        final_val_loss = float(
            history.history["val_loss"][-1]
        )

        epochs_completed = len(
            history.history["loss"]
        )

        accuracy_gain = (
            best_val_accuracy
            -
            float(
                history.history["val_accuracy"][0]
            )
        )
        
        generalization_gap = (
            final_train_accuracy
            -
            final_val_accuracy
        )
        
        overfitting_flag = (
            generalization_gap > 0.20
        )
        
        best_epoch_fraction = (
            best_epoch
            /
            epochs_completed
        )

        summary_row = {

            # -----------------------------
            # Experiment metadata
            # -----------------------------

            "model":
                MODEL_NAME,

            "seed":
                seed,

            "duration":
                duration,

            # -----------------------------
            # Architecture metadata
            # -----------------------------

            "learning_rate":
                learning_rate,

            # -----------------------------
            # Training metadata
            # -----------------------------

            "epochs_requested":
                EPOCHS,

            "epochs_completed":
                epochs_completed,

            "best_epoch":
                best_epoch,

            "best_epoch_fraction":
                best_epoch_fraction,

            "training_seconds":
                training_seconds,

            # -----------------------------
            # Validation metrics
            # -----------------------------

            "best_val_accuracy":
                best_val_accuracy,

            "best_val_loss":
                best_val_loss,

            "final_val_accuracy":
                final_val_accuracy,

            "final_val_loss":
                final_val_loss,

            "accuracy_gain":
                accuracy_gain,

            # -----------------------------
            # Training metrics
            # -----------------------------

            "final_train_accuracy":
                final_train_accuracy,

            "final_train_loss":
                final_train_loss,

            # -----------------------------
            # Generalization
            # -----------------------------

            "generalization_gap":
                generalization_gap,
            
            "overfitting_flag":
                overfitting_flag,
            
            # -----------------------------
            # Test metrics
            # -----------------------------

            "test_accuracy":
                metrics["accuracy"],

            "test_precision":
                metrics["precision"],

            "test_recall":
                metrics["recall"],

            "test_f1":
                metrics["f1"]
        }

        summary_results.append(
            summary_row
        )

        all_summary_rows.append(
            summary_row
        )

        # -----------------------------------------
        # Append duration results
        # -----------------------------------------

        results.append({

            k: v

            for k, v in metrics.items()

            if k not in [
                "classification_report",
                "confusion_matrix"
            ]
        })

    # ---------------------------------------------
    # Save seed-specific duration results
    # ---------------------------------------------

    pd.DataFrame(
        results
    ).to_csv(
        paths["duration_results"] /
        f"duration_results_seed_{seed}.csv",
        index=False
    )

    # ---------------------------------------------
    # Save seed-specific experiment summary
    # ---------------------------------------------

    pd.DataFrame(
        summary_results
    ).to_csv(
        paths["duration_results"] /
        f"experiment_summary_seed_{seed}.csv",
        index=False
    )

    print(
        f"\nCompleted seed {seed}"
    )

# =====================================================
# GLOBAL SUMMARY
# =====================================================

pd.DataFrame(
    all_summary_rows
).to_csv(
    paths["duration_results"] /
    "experiment_summary_all.csv",
    index=False
)

print(
    "\nSaved experiment_summary_all.csv"
)

In [ ]:
from collections import Counter

y_pred = []

for images, labels in test_ds:

    preds = model.predict(
        images,
        verbose=0
    )

    y_pred.extend(
        np.argmax(preds, axis=1)
    )

print(Counter(y_pred))